In [73]:
from cobra.io import read_sbml_model, write_sbml_model
from cobra.core import Reaction
from cobra.flux_analysis import flux_variability_analysis
import json

In [42]:
model = read_sbml_model("Recon2.2_FBC_gene_names_fixed.xml")

rxn_dict = {"atp_c": -1, "h2o_c": -1, "adp_c": 1, "h_c": 1, "pi_c": 1}
ATPM = Reaction("ATPM")
model.add_reactions([ATPM])
rxn_dict_cobra = {}
for m,s in rxn_dict.items():
    m = model.metabolites.get_by_id(m)
    rxn_dict_cobra[m] = s

ATPM.add_metabolites(rxn_dict_cobra)
ATPM.lower_bound = 8.6

In [53]:
media_dict = {}
with open("RPMI_1640_media.json") as fh:
    media_dict = json.load(fh)

for r in model.exchanges:
    if r.lower_bound == 0:
        continue
    r.lower_bound = 0

for k,v in media_dict.items():
    if k not in model.reactions:
        print(k)
        continue
    rxn = model.reactions.get_by_id(k)
    rxn.lower_bound = -v


model.reactions.EX_lac_L_e.upper_bound = 0
model.reactions.EX_for_e.upper_bound = 0
model.reactions.EX_thf_e.upper_bound = 0
model.reactions.EX_fe2_e.upper_bound = 0
model.reactions.EX_cys_L_e.upper_bound = 0
model.reactions.EX_gt1a_hs_e.upper_bound = 0
model.reactions.EX_4abut_e.upper_bound = 0
model.reactions.EX_mthgxl_e.upper_bound = 0
model.reactions.EX_bhb_e.upper_bound = 0
model.reactions.EX_ala_L_e.upper_bound = 0
model.reactions.EX_ala_D_e.upper_bound = 0


EX_zn2_e


In [55]:
#ATPM.lower_bound = 1000
model.reactions.EX_o2_e.lower_bound = -20
solution = model.optimize()
print(solution.fluxes["ATPM"])
model.summary()

8.6


Metabolite,Reaction,Flux,C-Number,C-Flux
arg_L_e,EX_arg_L_e,0.05,6,4.39%
asn_L_e,EX_asn_L_e,0.0236,4,1.38%
chol_e,EX_chol_e,0.005,5,0.37%
glc_D_e,EX_glc_e,0.2031,6,17.85%
gln_L_e,EX_gln_L_e,0.4206,5,30.81%
glu_L_e,EX_glu_L_e,0.05,5,3.66%
gthrd_e,EX_gthrd_e,0.05,10,7.32%
his_L_e,EX_his_L_e,0.03237,6,2.85%
ile_L_e,EX_ile_L_e,0.05,6,4.39%
inost_e,EX_inost_e,0.001969,6,0.17%


In [59]:
fva_df = flux_variability_analysis(model, processes=12)

In [64]:
eps = 1e-9
mask = (fva_df['minimum'].abs() < eps) & (fva_df['maximum'].abs() < eps)
blocked_reactions = fva_df[mask].index

In [70]:
blocked_reactions_set = set(blocked_reactions)
gap_metabolites = []
for m in model.metabolites:
    m_rxns = {r.id for r in m.reactions}
    if len(m_rxns - blocked_reactions_set) == 0:
        gap_metabolites.append(m.id)

In [72]:
print(f"N rxns: {len(model.reactions)}")
print(f"N mets: {len(model.metabolites)}")

trimmed = model.copy()
trimmed.remove_reactions(blocked_reactions)
gap_metabolites = [trimmed.metabolites.get_by_id(m) for m in gap_metabolites]
trimmed.remove_metabolites(gap_metabolites)

print(f"N rxns: {len(trimmed.reactions)}")
print(f"N mets: {len(trimmed.metabolites)}")

N rxns: 7786
N mets: 5324
Read LP format model from file /tmp/tmp6kap7l0f.lp
Reading time = 0.02 seconds
: 5324 rows, 15572 columns, 54984 nonzeros
N rxns: 4786
N mets: 2570


In [76]:
trimmed.optimize()
trimmed.summary()

write_sbml_model(trimmed, "./Recon2.2_FBC_trimmed_final.xml")

In [77]:
test = read_sbml_model("Recon2.2_FBC_trimmed_final.xml")
test.optimize()
test.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
arg_L_e,EX_arg_L_e,0.05,6,4.39%
asn_L_e,EX_asn_L_e,0.0236,4,1.38%
chol_e,EX_chol_e,0.005,5,0.37%
glc_D_e,EX_glc_e,0.2031,6,17.85%
gln_L_e,EX_gln_L_e,0.4206,5,30.81%
glu_L_e,EX_glu_L_e,0.05,5,3.66%
gthrd_e,EX_gthrd_e,0.05,10,7.32%
his_L_e,EX_his_L_e,0.03237,6,2.85%
ile_L_e,EX_ile_L_e,0.05,6,4.39%
inost_e,EX_inost_e,0.001969,6,0.17%


In [82]:
test.reactions.EX_o2_e.lower_bound = 0
test.optimize()
test.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
arg_L_e,EX_arg_L_e,0.05,6,0.83%
asn_L_e,EX_asn_L_e,0.05,4,0.55%
asp_L_e,EX_asp_L_e,0.05,4,0.55%
chol_e,EX_chol_e,0.005,5,0.07%
cys_L_e,EX_cys_L_e,0.05,3,0.41%
glc_D_e,EX_glc_e,5,6,82.94%
gln_L_e,EX_gln_L_e,0.5,5,6.91%
glu_L_e,EX_glu_L_e,0.05,5,0.69%
gly_e,EX_gly_e,0.05,2,0.28%
gthrd_e,EX_gthrd_e,0.05,10,1.38%
